# AHC015 128-channel PPO continuation from last (10 hours)

Save & Run All用Notebook。`large-20260905-093954`のlast training checkpoint（epoch 320）から、未来列なしafterstate 128 channelモデルを蒸留係数0で10時間PPO継続学習する。potential shapingは維持し、policy Phi係数は0。KaggleでGPU T4 x2、Internet Onを選び、`GITHUB_TOKEN`と`WANDB_API_KEY`のSecret accessを有効にする。rollout・評価はGPUごとの独立process、PPO更新はDDP/NCCLで2 GPUを使う。本番前に同じresume経路を8局・1 iterationで検証する。W&B run名は`large-<時刻>`。

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA GPU is required"
assert torch.cuda.device_count() == 2, "Select the Kaggle GPU T4 x2 accelerator"
for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    capability = torch.cuda.get_device_capability(index)
    print(index, name, capability)
    assert "T4" in name and capability == (7, 5), "GPU T4 x2 is required"
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
assert not repo_dir.exists(), f"Clean session required: {repo_dir} already exists"
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"
try:
    subprocess.run(
        ["git", "clone", "--branch", "feature/ahc015-teacher",
         "--single-branch", "https://github.com/e1jirou/ahc-ml.git", str(repo_dir)],
        check=True, env=git_env,
    )
finally:
    del github_token, credentials, git_env
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
os.chdir(repo_dir)
print("Repository commit:", actual_commit)
print("Current directory:", os.getcwd())

In [ ]:
%pip install --quiet torchview==0.2.7

from importlib.metadata import version
print("torchview:", version("torchview"))

In [ ]:
import hashlib

import wandb

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
del wandb_api_key
assert wandb.login(verify=True)
api = wandb.Api()
source_run = api.run("eijirou-personal/ahc-ml/ppf35jxs")
assert source_run.name == "large-20260905-093954"
assert source_run.state == "finished"
artifact_name = (
    "eijirou-personal/ahc-ml/"
    "large-20260905-093954-last-training-checkpoint:v0"
)
checkpoint_dir = Path("/kaggle/working/checkpoints") / source_run.name
artifact = api.artifact(artifact_name, type="model")
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
checkpoint_path = downloaded_dir / "last.pt"
assert checkpoint_path.is_file()
checkpoint_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
assert checkpoint_sha256 == "55c1326fe178b4e39063fa0344112c3320c6610c79b4674aed624475a96087ff"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
assert checkpoint["format_version"] == 1
assert checkpoint["epoch"] == 320
assert checkpoint["optimizer_state_dict"]["state"]
assert checkpoint["config"]["model"]["channels"] == 128
assert checkpoint["config"]["model"]["input_mode"] == "afterstate"
assert checkpoint["config"]["model"]["future_mode"] == "none"
print("Source W&B run:", source_run.name, source_run.id)
print("Checkpoint:", checkpoint_path)
print("Epoch:", checkpoint["epoch"])
print("SHA-256:", checkpoint_sha256)
print("Source last mean score:", checkpoint["metrics"]["evaluation/mean_score"])
print("Source best mean score:", checkpoint["metrics"]["evaluation/best_mean_score"])
del checkpoint

In [ ]:
import sys

sys.path.insert(0, str(repo_dir / "python"))
from examples.ahc015.python.config import load_config

config_path = repo_dir / "examples/ahc015/config_afterstate_128_continue.toml"
config = load_config(config_path)
assert config.run.name_prefix == "large"
assert config.run.seed == 15046
assert config.run.device == "cuda"
assert config.model.channels == 128 and config.model.residual_blocks == 10
assert config.model.input_mode == "afterstate"
assert config.model.future_mode == "none"
assert config.training.max_hours == 10.0
assert config.training.learning_rate == 3e-4
assert config.ppo.entropy_coefficient == 0.01
assert config.training.data_parallel and config.training.rollout_processes == 2
assert config.training.rollout_episodes == 4096
assert config.distillation.coefficient_start == 0.0
assert config.distillation.coefficient_end == 0.0
assert config.ppo.policy_phi_coefficient_start == 0.0
assert config.ppo.policy_phi_coefficient_end == 0.0
assert config.ppo.reward_mode == "potential_shaping"
assert config.evaluation.interval == 2 and config.evaluation.episodes == 2048
assert config.wandb.mode == "online"
print("Preflight passed:", config_path)

In [ ]:
train_env = os.environ.copy()
train_env["PYTHONPATH"] = str(repo_dir / "python")
smoke_output = Path("/kaggle/working/ahc015-large-smoke")
subprocess.run(
    [
        sys.executable, "-m", "examples.ahc015.python.train",
        "--config", str(config_path),
        "--resume", str(checkpoint_path),
        "--iterations", "322",
        "--max-hours", "0.1",
        "--rollout-episodes", "8",
        "--batch-size", "396",
        "--micro-batch-size", "396",
        "--evaluation-interval", "1",
        "--evaluation-episodes", "2",
        "--wandb-mode", "disabled",
        "--output-dir", str(smoke_output),
        "--experiment-log", "/kaggle/working/ahc015-large-smoke.md",
    ],
    cwd=repo_dir, env=train_env, check=True,
)
print("Multi-GPU resume smoke test passed; starting the 10-hour W&B run")
subprocess.run(
    [
        sys.executable, "-m", "examples.ahc015.python.train",
        "--config", str(config_path),
        "--resume", str(checkpoint_path),
    ],
    cwd=repo_dir, env=train_env, check=True,
)

In [ ]:
run_dirs = sorted((repo_dir / "outputs/ahc015").glob("large-*"))
assert run_dirs
latest_run = run_dirs[-1]
best_training_path = latest_run / "best-training.pt"
last_training_path = latest_run / "last.pt"
assert best_training_path.is_file() and last_training_path.is_file()
saved = torch.load(best_training_path, map_location="cpu", weights_only=False)
last_saved = torch.load(last_training_path, map_location="cpu", weights_only=False)
assert saved["config"]["model"]["future_mode"] == "none"
assert saved["config"]["model"]["channels"] == 128
assert saved["config"]["distillation"]["coefficient_start"] == 0.0
print("Completed run:", latest_run.name)
print("Last epoch:", last_saved["epoch"])
print("Final distillation coefficient:", last_saved["metrics"]["training/distillation_coefficient"])
print("Final mean score:", last_saved["metrics"]["evaluation/mean_score"])
last_artifact_name = f"eijirou-personal/ahc-ml/{latest_run.name}-last-training-checkpoint:latest"
uploaded_last_artifact = api.artifact(last_artifact_name, type="model")
assert any(file.name == "last.pt" for file in uploaded_last_artifact.files())
print("Best checkpoint:", best_training_path)
print("Last checkpoint:", last_training_path)
print("W&B last artifact:", uploaded_last_artifact.qualified_name)